# PatchQuestions
An offline benchmark for choosing the next test before committing to a patch. Run all cells on CPU. CPython 3.12.14 produced the reference values. All generated files go in a new working directory.

In [1]:
import json, os, sys, tempfile, runpy
from pathlib import Path
work = Path(tempfile.mkdtemp(prefix='patchquestions-', dir=Path.cwd())).resolve()
os.chdir(work)
sys.path.insert(0, str(work))
print('Python', sys.version.split()[0])

Python 3.12.14


In [2]:
%%writefile benchmark.py
"""Offline, executable API-contract recovery benchmark. Python >=3.11."""
import base64, binascii, csv, fnmatch, hashlib, io, itertools, json, math, random, re
import statistics, textwrap
from collections import Counter, defaultdict
from functools import lru_cache
from urllib.parse import parse_qsl, quote, quote_plus
from pathlib import Path

def product(*axes):
    return list(itertools.product(*axes))

def outcome(f, config, x):
    try:
        return json.dumps(["return", f(config,x)], ensure_ascii=True, sort_keys=True, allow_nan=False)
    except (ValueError, csv.Error, UnicodeError, binascii.Error) as exc:
        return json.dumps(["raise",type(exc).__name__])

def families():
    # Each row contains candidate configurations, one smoke input, finite universe, implementation.
    return {
        "url_quote": (product([0,1],["","/","/:","@","/@"],["strict","replace"]), "cat",
            [a+b for a,b in product(["hi","a b","/x","@",":","\ud800"],[""," b","/x","@y",":z","é","+"])],
            lambda c,x:(quote,quote_plus)[c[0]](x,safe=c[1],errors=c[2])),
        "query_pairs": (product([False,True],[False,True],["&",";"],["latin1","utf8"]), "a=1",
            [a+b for a,b in product(["a=1","b=","bare","x=%C3%A9","x=%FF",""],["","&q",";b=",";x=%FF","&b=2","+%2B"])],
            lambda c,x:parse_qsl(x,keep_blank_values=c[0],strict_parsing=c[1],separator=c[2],encoding=c[3])),
        "csv_fields": (product([",",";","\t"],[False,True],[False,True],[None,"\\"]), "x",
            [a+d+b for a,d,b in product(['x','"a"','"a""b"','a\\'],[',',';','\t'],[' y','"x','z','\\q'])],
            lambda c,x:list(csv.reader([x],delimiter=c[0],skipinitialspace=c[1],strict=c[2],escapechar=c[3]))),
        "wrap": (product([6,9],[False,True],[False,True],[False,True]), "hi",
            [a+s+b for a,s,b in product(["ab","abcdefgh","a-bcd","abcde"],[" ","  ","-","\t"],["z","efghijk","abc-def"])],
            lambda c,x:textwrap.wrap(x,width=c[0],break_long_words=c[1],break_on_hyphens=c[2],drop_whitespace=c[3])),
        "json_wire": (product([False,True],[False,True],[False,True],[False,True]), "1",
            [json.dumps({a:v,"a":w},ensure_ascii=False) for a,v,w in product(["z","é","B"],["é",[1,2],0],["x",["雪"],{}])] + ['[]','{}','"é"','[1, 2]'],
            lambda c,x:json.dumps(json.loads(x),ensure_ascii=c[0],sort_keys=c[1],separators=((',' if c[2] else ', '),(':' if c[3] else ': ')))),
        "regex_search": (product([0,re.I],[0,re.M],[0,re.S],[0,re.A]), "a|a",
            [p+'|'+s for p,s in product(["a","^b","a.b",r"\w+",r"\d+","[a-z]+","^a.b"],["A","a\nb","x\nb","é","９","aXb","K","!x"])],
            lambda c,x:[m.span() for m in re.finditer(x.split('|')[0],x.split('|')[1],flags=int(c[0]|c[1]|c[2]|c[3]))]),
        "base64_read": (product([None,b'-_',b'._'],[False,True]), "YQ==",
            [a+b for a,b in product(["AA","-_","._","+/","a!","//","ab","YWJj"],["==","/w==","\n==","!","_w==","", "==x"])],
            lambda c,x:base64.b64decode(x,altchars=c[0],validate=c[1]).hex()),
        "hex_format": (product([":","-"," "],[-3,-2,-1,1,2,3],[False,True]), "",
            [bytes((v+i)%256 for i in range(k)).hex() for k,v in product(range(1,12),[0,10,120,230])],
            lambda c,x:bytes.fromhex(x).hex(c[0],c[1]).upper() if c[2] else bytes.fromhex(x).hex(c[0],c[1]))
    }

def signature(row, cols):
    return tuple(row[j] for j in cols)

def evaluate(name, spec, seed, replacements=(), transfer_inputs=None):
    configs, smoke, inputs, f=spec
    inputs=sorted(set(inputs)-{smoke})
    random.Random(seed).shuffle(inputs)
    probes,hold=inputs[:min(16,len(inputs)//2)],inputs[min(16,len(inputs)//2):]
    if replacements:
        assert len(set(replacements)) == len(replacements)
        assert not set(replacements).intersection(inputs+[smoke])
        probes=probes[:-len(replacements)]+list(replacements)
    if transfer_inputs is not None:
        assert not set(transfer_inputs).intersection(probes+[smoke])
        hold=list(transfer_inputs)
    smokevals=[outcome(f,c,smoke) for c in configs]
    query=[tuple(outcome(f,c,x) for x in probes) for c in configs]
    hidden=[tuple(outcome(f,c,x) for x in hold) for c in configs]
    @lru_cache(None)
    def partition(ids,j):
        bins=defaultdict(list)
        for i in ids: bins[query[i][j]].append(i)
        return tuple(tuple(v) for v in bins.values())
    @lru_cache(None)
    def tree(ids,depth,frugal):
        # Maximize leaf count, optionally breaking ties by total path length over targets.
        best=(1,0,None)
        if depth==0: return best
        for j in range(len(probes)):
            parts=partition(ids,j)
            if len(parts)<2: continue
            children=[tree(p,depth-1,frugal) for p in parts]
            leaves=sum(child[0] for child in children)
            cost=len(ids)+sum(child[1] for child in children)
            if leaves>best[0] or (frugal and leaves==best[0] and cost<best[1]):
                best=(leaves,cost,j)
        return best
    methods=['random','random_split','diversity','entropy','lookahead','frugal']
    rows=[]
    for target in range(len(configs)):
        initial=tuple(i for i in range(len(configs)) if smokevals[i]==smokevals[target])

        @lru_cache(None)
        def score_state(ids, available, depth, method):
            if depth:
                if method=='random':
                    choices=available
                else:
                    informative=tuple(j for j in available if len(partition(ids,j))>1)
                    if method=='random_split':
                        choices=informative
                    elif method in ('lookahead','frugal'):
                        _,_,j=tree(ids,depth,method=='frugal')
                        choices=() if j is None else (j,)
                    else:
                        def utility(j):
                            sizes=[len(p) for p in partition(ids,j)]
                            if method=='diversity':
                                return len(sizes)
                            return -sum(n*math.log2(n) for n in sizes)
                        choices=(max(informative,key=utility),) if informative else ()
                if choices:
                    results=[]
                    for j in choices:
                        survivors=next(p for p in partition(ids,j) if target in p)
                        remaining=tuple(k for k in available if k!=j)
                        results.append(score_state(survivors,remaining,depth-1,method))
                    held,probe,count,used=map(statistics.fmean,zip(*results))
                    return held,probe,count,used+1
            return (sum(hidden[i]==hidden[target] for i in ids)/len(ids),
                    sum(query[i]==query[target] for i in ids)/len(ids),len(ids),0)
        for method in methods:
            for budget in [0,1,2,3]:
                metrics=score_state(initial,tuple(range(len(probes))),budget,method)
                rows.append([name,seed,target,method,budget,*metrics])
        score_state.cache_clear()
    tree.cache_clear(); partition.cache_clear()
    return rows

def run():
    rows=[]
    for name,spec in families().items():
        for seed in [19,43,73,101,137]:
            rows.extend(evaluate(name,spec,seed))
    return rows

def summarize(rows):
    grouped=defaultdict(list)
    for family,seed,target,method,budget,held,probe,n,used in rows:
        grouped[family,method,budget].append(held)
    means={k:statistics.mean(v) for k,v in grouped.items()}
    return {m:{b:statistics.mean(means[f,m,b] for f in families()) for b in [0,1,2,3]}
            for m in ['random','random_split','diversity','entropy','lookahead','frugal']},means

if __name__=='__main__':
    rows=run()
    with open('scores.csv','w',newline='') as out:
        w=csv.writer(out);w.writerow(['family','split','target','policy','budget','heldout_exact','probe_exact','survivors','queries_used']);w.writerows(rows)
    print(json.dumps(summarize(rows)[0],indent=2))


Writing benchmark.py


In [3]:
%%writefile replay.py
"""Blinded JSON protocol for one episode: python replay.py csv_fields 19 7."""
import argparse
import json
import random
from benchmark import families, outcome

def episode(family, split, target, seed):
    configs, smoke, universe, impl=families()[family]
    if not 0<=target<len(configs):
        raise ValueError('target outside family')
    xs=sorted(set(universe)-{smoke})
    random.Random(split).shuffle(xs)
    cut=min(16,len(xs)//2)
    probes,holdout=xs[:cut],xs[cut:]
    value=outcome(impl,configs[target],smoke)
    ids=[i for i,c in enumerate(configs) if outcome(impl,c,smoke)==value]
    random.Random(seed).shuffle(ids)
    grid=[[outcome(impl,configs[i],x) for x in probes] for i in ids]
    target_row=ids.index(target)
    return ids,probes,holdout,grid,target_row,configs,impl

def play(family, split, target, seed):
    ids,xs,held,grid,key,configs,impl=episode(family,split,target,seed)
    print(json.dumps(dict(family=family,inputs=xs,outputs=[[json.loads(o) for o in row] for row in grid],
                          budget=3,instructions='Send {"ask": column} or {"pick": row}; indices start at zero.')),flush=True)
    seen=set()
    for _ in range(4):
        action=json.loads(input())
        if set(action)=={'pick'}:
            row=action['pick']
            if type(row)!=int or not 0<=row<len(ids): raise ValueError('invalid row')
            correct=all(outcome(impl,configs[ids[row]],x)==outcome(impl,configs[target],x) for x in held)
            print(json.dumps(dict(heldout_exact=correct,queries=len(seen))),flush=True)
            return
        j=action.get('ask')
        if set(action)!={'ask'} or type(j)!=int or not 0<=j<len(xs) or j in seen or len(seen)>=3:
            raise ValueError('invalid or repeated query, or budget exceeded')
        seen.add(j)
        print(json.dumps(dict(answer=json.loads(grid[key][j]))),flush=True)

if __name__=='__main__':
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('family',choices=sorted(families()));p.add_argument('split',type=int)
    p.add_argument('target',type=int);p.add_argument('--shuffle',type=int,default=2026)
    a=p.parse_args();play(a.family,a.split,a.target,a.shuffle)


Writing replay.py


In [4]:
%%writefile analyze.py
"""Summarize exact expectations and compute the all-probes ceiling."""
import csv, statistics, json
from benchmark import families, outcome
from replay import episode

def audit():
    rows=list(csv.DictReader(open('scores.csv')))
    methods=list(dict.fromkeys(r['policy'] for r in rows))
    def family_mean(f,m,b,field):
        return statistics.fmean(float(r[field]) for r in rows if r['family']==f and r['policy']==m and int(r['budget'])==b)
    report={}
    for f,spec in families().items():
        vals=[]
        for seed in [19,43,73,101,137]:
            for t in range(len(spec[0])):
                ids,_,h,grid,key,c,impl=episode(f,seed,t,0)
                survivors=[i for i,row in zip(ids,grid) if row==grid[key]]
                vals.append(statistics.fmean(all(outcome(impl,c[i],x)==outcome(impl,c[t],x) for x in h) for i in survivors))
        report[f]={m:[family_mean(f,m,3,'heldout_exact'),family_mean(f,m,3,'queries_used')] for m in methods}
        report[f]['ceiling']=statistics.fmean(vals)
    return report

if __name__=='__main__':
    print(json.dumps(audit(),indent=2))


Writing analyze.py


In [5]:
%%writefile intervention.py
"""Post-hoc regex probe-design intervention. All arms share identical hidden inputs."""
import json
from benchmark import evaluate, families
from statistics import fmean
single=['q|Q','^q|p\nq','q.z|q\nz',r'\w|Ж']
compound=['^q.\\w|q0a\nQ0b\nq\nЖ\nq0é\nQ\nz','^q.\\w|Q0Ж\nq\nx\nq0b',
          '^q.\\w|q\nЖ\nQ0b\nq1Ж','^q.\\w|Q0b\nq\ny\nq1Ж']
def transfer_suite():
    # Separate vocabulary, strings and patterns; fixed after the diagnostic analysis.
    return [p+'|'+s for p in [r"^r.\w",r"r.\d","^r","r.t",r"\w+",r"\d+"]
            for s in ["R1Ж","r\nЖ","u\nr0a","Ｒ","r0۷","r\nt","!R0z","r0x\nR1Ж"]]

def study(transfer=False):
    out={}
    for label,probes in [('original',[]),('single',single),('compound',compound)]:
        rows=[r for seed in [19,43,73,101,137] for r in evaluate('regex_search',families()['regex_search'],seed,probes,transfer_suite() if transfer else None)]
        out[label]={b:fmean(r[5] for r in rows if r[3]=='frugal' and r[4]==b) for b in (1,2,3)}
    return out
if __name__=='__main__':
    print(json.dumps({"original_holdout":study(),"transfer":study(True)},indent=2))


Writing intervention.py


In [6]:
%%writefile verify.py
"""Independent decision-tree and blind-grader checks."""
import csv, json, subprocess, sys
from fractions import Fraction
from itertools import combinations
from benchmark import families, evaluate
from replay import episode
from intervention import study

def check():
    # Exhaustive two-query search using explicit leaf lists, independently of the DP.
    for name in families():
        ids,xs,held,grid,key,c,impl=episode(name,19,0,99)
        assert len(ids)==len(c)
        best=0
        for j in range(len(xs)):
            groups={}
            for i,row in enumerate(grid):
                groups.setdefault(row[j],[]).append(i)
            leaves=sum(max(len({grid[i][k] for i in group}) for k in range(len(xs))) for group in groups.values())
            best=max(best,leaves)
        # Uniform selection in each leaf scores exactly one correct identity per leaf.
        out=evaluate(name,families()[name],19)
        from math import isclose
        vals=[1/r[7] for r in out if r[3]=='frugal' and r[4]==2]
        assert isclose(sum(vals),best),name
        for b in (1,2,3):
            a=[r for r in out if r[3]=='lookahead' and r[4]==b]
            z=[r for r in out if r[3]=='frugal' and r[4]==b]
            assert isclose(sum(1/r[7] for r in a),sum(1/r[7] for r in z))
            assert sum(r[8] for r in z)<=sum(r[8] for r in a)+1e-9
    ids,_,_,_,key,_,_=episode('csv_fields',19,7,2026)
    cmd=[sys.executable,'replay.py','csv_fields','19','7']
    result=subprocess.run(cmd,input='{"ask":0}\n'+json.dumps({'pick':key})+'\n',capture_output=True,text=True,check=True)
    assert json.loads(result.stdout.splitlines()[-1])=={'heldout_exact':True,'queries':1}
    print('Eight independent optimality checks and blind protocol check passed.')

if __name__=='__main__':check()


Writing verify.py


In [7]:
%%writefile export_cases.py
"""Export exact query/held-out matrices for replay across Python versions."""
import gzip,json,random
from benchmark import families,outcome
def export(path):
    with gzip.open(path,'wt',encoding='utf8') as dst:
        for f,(configs,smoke,universe,impl) in families().items():
            for seed in [19,43,73,101,137]:
                xs=sorted(set(universe)-{smoke})
                random.Random(seed).shuffle(xs)
                cut=min(16,len(xs)//2)
                def matrix(inputs):
                    return [[json.loads(outcome(impl,c,x)) for x in inputs] for c in configs]
                record=dict(family=f,split=seed,smoke=smoke,configs=[list(map(repr,c)) for c in configs],
                            query_inputs=xs[:cut],query_outputs=matrix(xs[:cut]),
                            heldout_inputs=xs[cut:],heldout_outputs=matrix(xs[cut:]))
                dst.write(json.dumps(record,ensure_ascii=True)+'\n')
if __name__=='__main__': export('frozen_cases.jsonl.gz')


Writing export_cases.py


## Run the main experiment
Each row in scores.csv records one target, split, policy and budget. The printed summary weights families equally.

In [8]:
runpy.run_path('benchmark.py', run_name='__main__')

{
  "random": {
    "0": 0.07051504629629629,
    "1": 0.3071941912615741,
    "2": 0.46353822544642853,
    "3": 0.5683128759984228
  },
  "random_split": {
    "0": 0.07051504629629629,
    "1": 0.34244809737908694,
    "2": 0.5920951422795265,
    "3": 0.7631887749141841
  },
  "diversity": {
    "0": 0.07051504629629629,
    "1": 0.5208854166666667,
    "2": 0.7279166666666667,
    "3": 0.848125
  },
  "entropy": {
    "0": 0.07051504629629629,
    "1": 0.5163541666666667,
    "2": 0.7288541666666667,
    "3": 0.8527083333333333
  },
  "lookahead": {
    "0": 0.07051504629629629,
    "1": 0.5208854166666667,
    "2": 0.7442708333333333,
    "3": 0.8547916666666666
  },
  "frugal": {
    "0": 0.07051504629629629,
    "1": 0.5208854166666667,
    "2": 0.7442708333333333,
    "3": 0.8547916666666666
  }
}


## Diagnose the available probes
Compare three questions with asking for every available label. The latter reveals when a better selection policy has little room to help.

In [9]:
from analyze import audit
report=audit()
print('Family          Entropy   Frugal   All probes')
for family, row in report.items():
    print(f"{family:15s} {row['entropy'][0]:7.1%} {row['frugal'][0]:8.1%} {row['ceiling']:10.1%}")

Family          Entropy   Frugal   All probes
url_quote         98.0%    98.0%     100.0%
query_pairs       95.0%    95.0%      95.0%
csv_fields        83.3%    85.0%      85.0%
wrap              92.5%    92.5%      92.5%
json_wire        100.0%   100.0%     100.0%
regex_search      30.0%    30.0%      30.0%
base64_read       83.3%    83.3%      83.3%
hex_format       100.0%   100.0%     100.0%


## Change the regex inputs
The four replacements were designed after observing the regex failure. Both the original held-out inputs and the secondary transfer suite are isolated from the query policy.

In [10]:
runpy.run_path('intervention.py', run_name='__main__')

{
  "original_holdout": {
    "original": {
      "1": 0.15,
      "2": 0.225,
      "3": 0.3
    },
    "single": {
      "1": 0.15,
      "2": 0.25,
      "3": 0.5
    },
    "compound": {
      "1": 0.51,
      "2": 0.75,
      "3": 0.8625
    }
  },
  "transfer": {
    "original": {
      "1": 0.125,
      "2": 0.225,
      "3": 0.3
    },
    "single": {
      "1": 0.125,
      "2": 0.25,
      "3": 0.5
    },
    "compound": {
      "1": 0.5,
      "2": 0.75,
      "3": 0.8625
    }
  }
}


## Verify the reference solver and export matrices
The verification compares depth-two tree performance to independent explicit enumeration for every family and exercises the interactive grader.

In [11]:
runpy.run_path('verify.py', run_name='__main__')
runpy.run_path('export_cases.py', run_name='__main__')
print('Per-target results and frozen matrices are in', work.name)

Eight independent optimality checks and blind protocol check passed.
Per-target results and frozen matrices are in patchquestions-6bbppgmx
